# ESM2 as-standard model

Notebook draft to generate ESM2 embeddings from pylogeny-aware data. 

Tasks
- Import ESM checkpoint
- Import ESM tokenizer
- Import input data (target)
- Preprocess data
- Tokenize data
- Generate embeddings

Downstream tasks
- Run embeddings through classification head pre-trained using lower-level data for token classification
- Assess performance


In [1]:
# Dependancies and libraries
import torch
import esm
from transformers import AutoModel, AutoTokenizer
import pandas as pd

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer and model objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

df_target


,Info_protein_id,Info_pos,Info_AA,Info_group,Class
0,P24301.2,1,M,621.0,1.0
1,P24301.2,2,A,621.0,1.0
2,P24301.2,3,K,621.0,1.0
3,P24301.2,4,V,621.0,1.0
4,P24301.2,5,K,621.0,1.0
...,...,...,...,...,...
9016,O33084.3,96,S,622.0,-1.0
9017,O33084.3,97,K,622.0,-1.0
9018,O33084.3,98,M,622.0,-1.0
9019,O33084.3,99,N,622.0,-1.0


In [39]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [40]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [41]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [42]:
# Verify shape of embedding
outputs.last_hidden_state.size()


torch.Size([21, 1024, 320])

In [43]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

### Train classification head on lower level data

Tasks
- Load lower level data
- Preprocess
- Split into train/test
- Add grouped k fold train / val splits
- Instantiate classifier for token classification
- Train
- Test

In [54]:
# Import lower level data
df_lower= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Lower_1763.csv")

# Remove extraneous columns
df_lower = df_lower.iloc[:,0:5]

# Add mask column where 1 assigned if labelled with pos/neg epitope and -100 if NaN
df_lower['mask'] = df_lower['Class'].isin([-1,1]).astype('int32')
df_lower['mask'] = df_lower['mask'].replace(0, -100)

df_lower

,Info_protein_id,Info_pos,Info_AA,Info_group,Class,mask
0,P0A4V2.1,1,M,386.0,NaN,-100
1,P0A4V2.1,2,Q,386.0,NaN,-100
2,P0A4V2.1,3,L,386.0,NaN,-100
3,P0A4V2.1,4,V,386.0,NaN,-100
4,P0A4V2.1,5,D,386.0,NaN,-100
...,...,...,...,...,...,...
139122,YP_002644961.1,321,S,408.0,-1.0,1
139123,YP_002644961.1,322,L,408.0,-1.0,1
139124,YP_002644961.1,323,G,408.0,-1.0,1
139125,YP_002644961.1,324,A,408.0,-1.0,1


In [55]:
# Aggregate columns for wide format
df_lower = df_lower.groupby(['Info_protein_id','Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

df_lower

,,sequence,label,position
Info_protein_id,Info_group,,,
A1KFU9.1,544.0,MAENSNIDDIKAPLLAALGAADLALATVNELITNLRERAEETRTDT...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
A43589,594.0,MLGNAPSVVPNTTLGMHCGSFGSAPSNGWLKLGLVEFGGVAKLNAE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA21416.1,46.0,MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLRE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA21417.1,578.0,MLDVNFFDELRIGLATAEDIRQWSYGEVKKPETINYRTLKPEKDGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
AAA25359.1,410.0,MTDVSRKIRAWGRRLMIGTAAAVVLPGLVGLAGGAATAGAFSRPGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
...,...,...,...,...
YP_178023.1,632.0,MTEQQWNFAGIEAAASAIQGNVTSIHSLLDEGKQSLTKLAAAWGGS...,"[nan, nan, nan, -1.0, -1.0, -1.0, -1.0, -1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
YP_976577.1,178.0,MAKTIAYDEEARRGLERGLNALADAVKVTLGPKGRNVVLEKKWGAP...,"[nan, nan, nan, nan, nan, nan, 1.0, 1.0, 1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
ZP_03425930.1,172.0,MAEELHAAAGSFASVTTGLAGDAWHGPASLAMTRAASPYVGWLNTA...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."


In [56]:
# Check max length of sequences
df_lower['label'].str.len().agg(['mean','max'])

mean     402.427729
max     3186.000000
Name: label, dtype: float64

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data